# 01 · Quantum Foundations

**Covers:** quantum states, superposition, the Born rule, inner products, operators, eigenvalues, expectation values and the Bloch sphere.
**Photonics thread:** one photon's polarization. Every quantum state here is a Jones vector you already know from classical optics.

Each concept follows the same pattern: the idea in plain words, the honest gap in that explanation, a one-sentence summary, then code that checks the simulation against the analytic answer.

In [1]:
import numpy as np
import qutip as qt

results = []   # every theory-vs-simulation check is recorded here

def check(label, sim, theory, tol=1e-3):
    """Compare a simulated number with its analytic value and record the result."""
    sim, theory = float(np.real(sim)), float(np.real(theory))
    ok = bool(np.isclose(sim, theory, atol=tol))
    results.append(ok)
    print(f"{'PASS' if ok else 'FAIL'}  {label}: simulation = {sim:.4f}, theory = {theory:.4f}")

## 1. A quantum state is a Jones vector

**Plain words:** horizontal polarization is the list (1, 0), vertical is (0, 1). A quantum state of one photon is the same kind of list, written as a *ket* $|\psi\rangle$. We call horizontal $|0\rangle$ and vertical $|1\rangle$.

$$|\psi\rangle = a_0|0\rangle + a_1|1\rangle, \qquad |a_0|^2 + |a_1|^2 = 1$$

The entries $a_0, a_1$ are **amplitudes**: they can be negative or complex, and they are not probabilities.

**Gap:** a superposition is *not* "secretly H or V, we just don't know which". A photon at 45° passes a 45° polarizer every time; a photon that was secretly H or V would pass only half the time (shown with density matrices in notebook 05).

**One sentence:** a quantum state is a normalized list of complex amplitudes, exactly like a Jones vector.

In [2]:
H = qt.basis(2, 0)                 # |0> = horizontal
V = qt.basis(2, 1)                 # |1> = vertical
psi = qt.Qobj([[0.6], [0.8]])      # a polarization at about 53 degrees

print(psi.full().real.flatten())
check("norm of psi", psi.norm(), 1.0)

[0.6 0.8]
PASS  norm of psi: simulation = 1.0000, theory = 1.0000


## 2. Measurement: the Born rule

**Plain words:** we never see the state, only detector clicks. The probability of an outcome is the squared overlap of the state with that outcome:

$$P(\phi) = |\langle\phi|\psi\rangle|^2$$

Square first, then add: for (0.6, 0.8) the amplitudes sum to 1.4, but the probabilities 0.36 + 0.64 sum to 1.

**Optics link:** for a state at angle $\theta$, $P(H) = \cos^2\theta$. This is Malus's law, read for a single photon as a probability instead of a transmitted intensity.

**Gap:** the rule gives the odds, not which detector clicks on a given run. Physics has no deeper answer to that.

**One sentence:** probability = (shadow of the state on the outcome)².

In [3]:
check("P(H) for (0.6, 0.8)", abs(H.dag() * psi) ** 2, 0.36)
check("P(V) for (0.6, 0.8)", abs(V.dag() * psi) ** 2, 0.64)

plus = (H + V).unit()                                    # 45 degrees
check("P(45 deg) for a horizontal photon", abs(plus.dag() * H) ** 2, 0.5)

theta = np.deg2rad(30)                                   # Malus's law check
state = np.cos(theta) * H + np.sin(theta) * V
check("Malus: P(H) at 30 deg", abs(H.dag() * state) ** 2, np.cos(theta) ** 2)

PASS  P(H) for (0.6, 0.8): simulation = 0.3600, theory = 0.3600
PASS  P(V) for (0.6, 0.8): simulation = 0.6400, theory = 0.6400
PASS  P(45 deg) for a horizontal photon: simulation = 0.5000, theory = 0.5000
PASS  Malus: P(H) at 30 deg: simulation = 0.7500, theory = 0.7500


## 3. Inner product and complex amplitudes

**Plain words:** the inner product $\langle\phi|\psi\rangle$ measures how much one state overlaps another: 1 for the same state, 0 for orthogonal ones. The bra $\langle\phi|$ is the conjugated row version of the ket.

Complex numbers are needed because amplitudes carry a **phase**, and phase differences are what create interference. Circular polarization $(1, i)/\sqrt{2}$ is the standard example.

**Gap:** the complex conjugate looks fussy, but without it a complex state would not have a positive length.

**One sentence:** the inner product is the overlap, and its squared size is a probability.

In [4]:
check("<H|V> (orthogonal)", abs(H.dag() * V), 0.0)

circ = qt.Qobj([[1], [1j]]) / np.sqrt(2)                # circular polarization
check("<R|R> with conjugate", (circ.dag() * circ).real, 1.0)
no_conj = (circ.full().T @ circ.full())[0, 0]            # forgetting the conjugate
print("without the conjugate the 'length' would be", np.round(no_conj, 3))

PASS  <H|V> (orthogonal): simulation = 0.0000, theory = 0.0000
PASS  <R|R> with conjugate: simulation = 1.0000, theory = 1.0000
without the conjugate the 'length' would be 0j


## 4. Operators are Jones matrices

**Plain words:** an operator is a machine, state in and state out, just like a Jones matrix.

| Optical element | Matrix | Effect on (0.6, 0.8) |
|---|---|---|
| Horizontal polarizer | $P = \begin{pmatrix}1&0\\0&0\end{pmatrix}$ | (0.6, 0): V is absorbed |
| Half-wave plate at 45° | $X = \begin{pmatrix}0&1\\1&0\end{pmatrix}$ | (0.8, 0.6): H and V swapped |

**One sentence:** an operator is a Jones matrix by another name.

In [5]:
P = qt.Qobj([[1, 0], [0, 0]])
X = qt.sigmax()
print("polarizer:", (P * psi).full().real.flatten())
print("swap X:   ", (X * psi).full().real.flatten())
check("probability kept by X", (X * psi).norm() ** 2, 1.0)
check("probability kept by the polarizer", (P * psi).norm() ** 2, 0.36)

polarizer: [0.6 0. ]
swap X:    [0.8 0.6]
PASS  probability kept by X: simulation = 1.0000, theory = 1.0000
PASS  probability kept by the polarizer: simulation = 0.3600, theory = 0.3600


## 5. Eigenvectors, eigenvalues and Hermitian operators

**Plain words:** most states are changed by a machine. Its **eigenvectors** are only scaled, and the scale factor is the **eigenvalue**. For a measurement, the eigenvalues are the possible results and the eigenvectors are the states that give a result with certainty.

For the swap $X$, the 45° state $(1,1)/\sqrt2$ is unchanged (eigenvalue +1) and the 135° state $(1,-1)/\sqrt2$ flips sign (eigenvalue −1). So $X$ is the question "45° or 135°?".

Measurements must give real numbers, which is why they are **Hermitian** ($A = A^\dagger$): real eigenvalues, orthogonal eigenvectors.

**Gap:** an overall sign (−1 times the whole state) cannot be detected, so NumPy may print eigenvectors with flipped signs. Only *relative* signs matter.

**One sentence:** a measurement is a Hermitian matrix whose eigenvalues are the results and whose eigenvectors give them for sure.

In [6]:
vals, states = X.eigenstates()
print("eigenvalues of X:", vals)
check("X is Hermitian", float(X.isherm), 1.0)
check("P(45 deg) for psi", abs(states[1].dag() * psi) ** 2, (0.6 + 0.8) ** 2 / 2)
check("P(135 deg) for psi", abs(states[0].dag() * psi) ** 2, (0.6 - 0.8) ** 2 / 2)

eigenvalues of X: [-1.  1.]
PASS  X is Hermitian: simulation = 1.0000, theory = 1.0000
PASS  P(45 deg) for psi: simulation = 0.9800, theory = 0.9800
PASS  P(135 deg) for psi: simulation = 0.0200, theory = 0.0200


## 6. Expectation values and the Bloch sphere

**Plain words:** ask the same question of many identical photons and average the ±1 answers. That average is the expectation value $\langle A\rangle = \langle\psi|A|\psi\rangle$. No single photon ever gives −0.28; only the average does.

The three averages $(\langle X\rangle, \langle Y\rangle, \langle Z\rangle)$ form the **Bloch vector**. A pure state sits on the surface of the sphere (length 1). This is the Poincaré sphere from polarization optics.

**One sentence:** the expectation value is the average result, and three of them place the state on the Bloch sphere.

In [7]:
Y, Z = qt.sigmay(), qt.sigmaz()
ex, ey, ez = (qt.expect(op, psi) for op in (X, Y, Z))
check("<Z> = 0.36(+1) + 0.64(-1)", ez, -0.28)
check("<X> = 0.98(+1) + 0.02(-1)", ex, 0.96)
check("Bloch vector length (pure state)", np.sqrt(ex**2 + ey**2 + ez**2), 1.0)

PASS  <Z> = 0.36(+1) + 0.64(-1): simulation = -0.2800, theory = -0.2800
PASS  <X> = 0.98(+1) + 0.02(-1): simulation = 0.9600, theory = 0.9600
PASS  Bloch vector length (pure state): simulation = 1.0000, theory = 1.0000


In [8]:
import matplotlib.pyplot as plt
b = qt.Bloch()
b.add_states([H, V, plus, circ, psi])
b.render()
plt.show()

## Validation summary

Every `check` above compared a QuTiP result with its analytic formula.

In [9]:
print(f"{sum(results)} of {len(results)} checks passed")
assert all(results), "a simulation disagrees with theory"

15 of 15 checks passed
